In [1]:
import pandas as pd
import numpy as np

### Load tournament results

In [2]:
tourney_men_df = pd.read_csv("../data/MNCAATourneyCompactResults.csv")
tourney_women_df = pd.read_csv("../data/WNCAATourneyCompactResults.csv")

- Because the season data from `_RegularSeasonDetailedResults.csv` goes from 2003 -> 2026 for men and 2010 -> 2025 for women, we need to filter out the dataframe to correspond to these years

In [3]:
tourney_men_df = tourney_men_df[tourney_men_df["Season"] >= 2003].reset_index(drop=True)
tourney_women_df = tourney_women_df[tourney_women_df["Season"] >= 2010].reset_index(drop=True)

In [4]:
tourney_men_df.head(5)

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
0,2003,134,1421,92,1411,84,N,1
1,2003,136,1112,80,1436,51,N,0
2,2003,136,1113,84,1272,71,N,0
3,2003,136,1141,79,1166,73,N,0
4,2003,136,1143,76,1301,74,N,1


### Build historical rows

In [5]:
men_hist = tourney_men_df.copy()
men_hist_rev = tourney_men_df.copy()

Now we need to convert each tournament game into 2 rows, where first row has the winner with a target of 1(winner) and the other team with the target of 0 (loser)

### Team1 wins

In [6]:
men_hist["Team1ID"] = men_hist["WTeamID"]
men_hist["Team2ID"] = men_hist["LTeamID"]
men_hist["Target"] = 1

### Team1 loses

In [7]:
men_hist_rev["Team1ID"] = men_hist_rev["LTeamID"]
men_hist_rev["Team2ID"] = men_hist_rev["WTeamID"]
men_hist_rev["Target"] = 0

Merge both datasets together

In [8]:
men_hist = pd.concat([men_hist, men_hist_rev], ignore_index=True)

In [9]:
men_hist = men_hist[["Season", "DayNum", "WLoc", "NumOT", "Team1ID", "Team2ID", "Target"]]
men_hist.head()

,Season,DayNum,WLoc,NumOT,Team1ID,Team2ID,Target
0,2003,134,N,1,1421,1411,1
1,2003,136,N,0,1112,1436,1
2,2003,136,N,0,1113,1272,1
3,2003,136,N,0,1141,1166,1
4,2003,136,N,1,1143,1301,1


### Now we do the same thing for women

In [10]:
women_hist = tourney_women_df.copy()
women_hist_rev = women_hist.copy()

In [11]:
women_hist["Team1ID"] = women_hist["WTeamID"]
women_hist["Team2ID"] = women_hist["LTeamID"]
women_hist["Target"] = 1

In [12]:
women_hist_rev["Team1ID"] = women_hist_rev["LTeamID"]
women_hist_rev["Team2ID"] = women_hist_rev["WTeamID"]
women_hist_rev["Target"] = 0

In [13]:
women_hist = pd.concat([women_hist, women_hist_rev], ignore_index=True)
women_hist = women_hist[["Season", "DayNum", "WLoc", "NumOT", "Team1ID", "Team2ID", "Target"]]

In [14]:
women_hist.head()

,Season,DayNum,WLoc,NumOT,Team1ID,Team2ID,Target
0,2010,138,N,0,3124,3201,1
1,2010,138,N,0,3173,3395,1
2,2010,138,H,0,3181,3214,1
3,2010,138,H,0,3199,3256,1
4,2010,138,N,0,3207,3265,1


### Merge with team_features

In [15]:
team_features_men = pd.read_csv("../data/m_team_season_features.csv")
team_features_women = pd.read_csv("../data/w_team_season_features.csv")

By separating the teams by winners and losers, it allows us to calculate all the different features between each team

In [16]:
def merge_team_features(matchups_df, team_features):
    df = matchups_df.copy()

    team1_features = team_features.add_prefix("Team1_")
    team2_features = team_features.add_prefix("Team2_")

    df = df.merge(
        team1_features,
        left_on=["Season", "Team1ID"],
        right_on=["Team1_Season", "Team1_TeamID"],
        how="left"
    )

    df = df.merge(
        team2_features,
        left_on=["Season", "Team2ID"],
        right_on=["Team2_Season", "Team2_TeamID"],
        how="left"
    )

    return df

In [17]:
men_matchups = merge_team_features(men_hist, team_features_men)
men_matchups.head()

,Season,DayNum,WLoc,NumOT,Team1ID,Team2ID,Target,Team1_Season,Team1_TeamID,Team1_Wins,...,Team2_AvgNetRating,Team2_AvgGameTotalPoints,Team2_Losses,Team2_WinPct,Team2_LossPct,Team2_ConfAbbrev,Team2_Seed,Team2_SeedNum,Team2_HasTournamentSeed,Team2_MasseyOrdinalRank
0,2003,134,N,1,1421,1411,1,2003,1421,13,...,0.020735,143.633333,12,0.600000,0.400000,swac,X16a,16.0,1,249.0
1,2003,136,N,0,1112,1436,1,2003,1112,25,...,0.074483,130.931034,10,0.655172,0.344828,aec,Z16,16.0,1,148.0
2,2003,136,N,0,1113,1272,1,2003,1113,18,...,0.124869,140.344828,6,0.793103,0.206897,cusa,Z07,7.0,1,18.0
3,2003,136,N,0,1141,1166,1,2003,1141,23,...,0.211501,143.575758,4,0.878788,0.121212,mvc,Z06,6.0,1,19.0
4,2003,136,N,1,1143,1301,1,2003,1143,21,...,0.064883,140.400000,12,0.600000,0.400000,acc,W09,9.0,1,48.0


In [18]:
women_matchups = merge_team_features(women_hist, team_features_women)
women_matchups.head()

,Season,DayNum,WLoc,NumOT,Team1ID,Team2ID,Target,Team1_Season,Team1_TeamID,Team1_Wins,...,Team2_AvgGameTotalPoints,Team2_Losses,Team2_WinPct,Team2_LossPct,Team2_ConfAbbrev,Team2_Seed,Team2_SeedNum,Team2_HasTournamentSeed,Team2_WMasseyRating,Team2_WMasseySOS
0,2010,138,N,0,3124,3201,1,2010,3124,23,...,134.030303,6,0.818182,0.181818,wac,X13,13.0,1,15.562954,2.684166
1,2010,138,N,0,3173,3395,1,2010,3173,21,...,132.600000,8,0.733333,0.266667,mwc,X09,9.0,1,16.548511,4.548511
2,2010,138,H,0,3181,3214,1,2010,3181,27,...,117.966667,11,0.633333,0.366667,meac,X15,15.0,1,0.058241,-7.641759
3,2010,138,H,0,3199,3256,1,2010,3199,25,...,138.580645,8,0.741935,0.258065,wac,W14,14.0,1,10.067330,0.131846
4,2010,138,N,0,3207,3265,1,2010,3207,24,...,126.090909,7,0.787879,0.212121,maac,X12,12.0,1,9.823899,-0.448829


### Feature engineering
- WinPctDiff
- WinPctGap
- SeedNumDiff
- SeedGap
- NetRatingDiff
- NetRatingGap

In [19]:
def add_matchup_features(df, rating_col=None):
    df = df.copy()

    # Win %
    df["WinPctDiff"] = df["Team1_WinPct"] - df["Team2_WinPct"]
    df["WinPctGap"] = np.abs(df["WinPctDiff"])

    # Seed
    df["SeedNumDiff"] = df["Team1_SeedNum"] - df["Team2_SeedNum"]
    df["SeedGap"] = np.abs(df["SeedNumDiff"])

    # Net Rating
    df["NetRatingDiff"] = df["Team1_AvgNetRating"] - df["Team2_AvgNetRating"]
    df["NetRatingGap"] = np.abs(df["NetRatingDiff"])

    # Efficiency
    df["OffEffDiff"] = df["Team1_AvgOffEfficiency"] - df["Team2_AvgOffEfficiency"]
    df["DefEffDiff"] = df["Team1_AvgDefEfficiency"] - df["Team2_AvgDefEfficiency"]

    # Margin
    df["MarginDiff"] = df["Team1_AvgMarginScore"] - df["Team2_AvgMarginScore"]

    # Rebounding
    df["ReboundPctDiff"] = df["Team1_AvgReboundPct"] - df["Team2_AvgReboundPct"]

    # Turnovers
    df["TurnoverPctDiff"] = df["Team1_AvgTurnoverPct"] - df["Team2_AvgTurnoverPct"]

    # Shooting
    df["FGPctDiff"] = df["Team1_AvgFieldGoalsPct"] - df["Team2_AvgFieldGoalsPct"]
    df["ThreePctDiff"] = df["Team1_AvgThreePointsPct"] - df["Team2_AvgThreePointsPct"]
    df["FTPctDiff"] = df["Team1_AvgFreeThrowPct"] - df["Team2_AvgFreeThrowPct"]

    # Ranking
    if rating_col is not None:
        df["RankingDiff"] = df[f"Team1_{rating_col}"] - df[f"Team2_{rating_col}"]
        df["RankingDiff"] = df["RankingDiff"].fillna(0)

    return df

In [20]:
men_matchups = add_matchup_features(men_matchups, rating_col="MasseyOrdinalRank")
women_matchups = add_matchup_features(women_matchups, rating_col="WMasseyRating")

In [35]:
men_matchups["PossessionsDiff"] = (
        men_matchups["Team1_AvgPossessions"] -
        men_matchups["Team2_AvgPossessions"]
)

men_matchups["ReboundMarginDiff"] = (
        men_matchups["Team1_AvgReboundMargin"] -
        men_matchups["Team2_AvgReboundMargin"]
)

men_matchups["TurnoverMarginDiff"] = (
        men_matchups["Team1_AvgTurnoverMargin"] -
        men_matchups["Team2_AvgTurnoverMargin"]
)

men_matchups["AssistTurnoverRatioDiff"] = (
        men_matchups["Team1_AvgAssistTurnoverRto"] -
        men_matchups["Team2_AvgAssistTurnoverRto"]
)



In [36]:
women_matchups["PossessionsDiff"] = (
        men_matchups["Team1_AvgPossessions"] -
        men_matchups["Team2_AvgPossessions"]
)

In [37]:
men_matchups = men_matchups.copy()

In [38]:
men_matchups.columns.tolist()

['Season',
 'DayNum',
 'WLoc',
 'NumOT',
 'Team1ID',
 'Team2ID',
 'Target',
 'Team1_Season',
 'Team1_TeamID',
 'Team1_Wins',
 'Team1_GamesPlayed',
 'Team1_AvgPointsFor',
 'Team1_AvgPointsAgainst',
 'Team1_AvgMarginScore',
 'Team1_MarginScoreStd',
 'Team1_AvgNumOT',
 'Team1_AvgFieldGoalsMade',
 'Team1_AvgFieldGoalsAttempted',
 'Team1_AvgFieldGoalsPct',
 'Team1_AvgThreePointsMade',
 'Team1_AvgThreePointsAttempted',
 'Team1_AvgThreePointsPct',
 'Team1_AvgFreeThrowsMade',
 'Team1_AvgFreeThrowsAttempted',
 'Team1_AvgFreeThrowPct',
 'Team1_AvgOffensiveRebounds',
 'Team1_AvgDefensiveRebounds',
 'Team1_AvgTotalRebounds',
 'Team1_AvgReboundPct',
 'Team1_AvgOffensiveReboundPct',
 'Team1_AvgDefensiveReboundPct',
 'Team1_AvgAssists',
 'Team1_AvgTurnovers',
 'Team1_AvgTurnoverPct',
 'Team1_AvgAssistTurnoverRto',
 'Team1_AvgSteals',
 'Team1_AvgBlocks',
 'Team1_AvgPersonalFouls',
 'Team1_AvgTurnoverMargin',
 'Team1_AvgReboundMargin',
 'Team1_AvgPossessions',
 'Team1_AvgOffEfficiency',
 'Team1_AvgDefE

### Build dataset for training

For training we need:
 - Team identifiers
  - seasons
  - target variable
  - engineered features

Right now we only have 12 features, but we can play around and see how we can add or remove them as we go. I believe we have a total of 100(?) features to use =)

### TODO Create women_training_df

In [39]:
men_training_df = men_matchups[
    [
        "Season",
        "Team1ID",
        "Team2ID",
        "Target",
        "WinPctDiff",
        "SeedNumDiff",
        "NetRatingDiff",
        "OffEffDiff",
        "DefEffDiff",
        "MarginDiff",
        "ReboundPctDiff",
        "TurnoverPctDiff",
        "FGPctDiff",
        "ThreePctDiff",
        "FTPctDiff",
        "RankingDiff",
        "PossessionsDiff",
        "TurnoverMarginDiff",
        "ReboundMarginDiff",
        "AssistTurnoverRatioDiff"

    ]
].copy()
men_training_df.head()

,Season,Team1ID,Team2ID,Target,WinPctDiff,SeedNumDiff,NetRatingDiff,OffEffDiff,DefEffDiff,MarginDiff,ReboundPctDiff,TurnoverPctDiff,FGPctDiff,ThreePctDiff,FTPctDiff,RankingDiff,PossessionsDiff,TurnoverMarginDiff,ReboundMarginDiff,AssistTurnoverRatioDiff
0,2003,1421,1411,1,-0.151724,0.0,-0.118699,-0.021731,0.096968,-9.208046,-0.030010,0.012414,-0.016123,0.042080,0.152397,16.0,0.251126,-2.479310,-4.270115,-0.066843
1,2003,1112,1436,1,0.237685,-15.0,0.120916,0.081979,-0.038936,10.309113,-0.013179,-0.019188,0.017191,-0.006859,0.051446,-145.0,10.753153,3.140394,-0.812808,0.216026
2,2003,1113,1272,1,-0.172414,3.0,-0.023829,0.040148,0.063977,-1.896552,0.010731,0.006264,0.042222,-0.015065,0.047369,22.0,-1.060690,0.241379,1.344828,-0.089779
3,2003,1141,1166,1,-0.085684,5.0,-0.126746,-0.040490,0.086256,-8.805643,0.007251,0.055825,0.008041,-0.007432,0.073034,17.0,2.667962,-5.869383,1.087774,-0.455413
4,2003,1143,1301,1,0.124138,-1.0,0.002024,-0.020811,-0.022835,0.324138,0.009767,-0.012174,0.010234,0.025371,-0.089516,-18.0,3.437425,0.325287,1.648276,0.128020


In [47]:
men_training_df["RankingDiff"] = men_training_df["RankingDiff"].fillna(0)
men_training_df["AssistTurnoverRatioDiff"] = men_training_df["AssistTurnoverRatioDiff"].fillna(0)

In [48]:
men_training_df.isna().sum().sort_values(ascending=False)

Season                     0
Team1ID                    0
Team2ID                    0
Target                     0
WinPctDiff                 0
SeedNumDiff                0
NetRatingDiff              0
OffEffDiff                 0
DefEffDiff                 0
MarginDiff                 0
ReboundPctDiff             0
TurnoverPctDiff            0
FGPctDiff                  0
ThreePctDiff               0
FTPctDiff                  0
RankingDiff                0
PossessionsDiff            0
TurnoverMarginDiff         0
ReboundMarginDiff          0
AssistTurnoverRatioDiff    0
dtype: int64

In [49]:
men_training_df["Target"].value_counts()

Target
1    1449
0    1449
Name: count, dtype: int64

In [50]:
missing_rows = men_matchups[men_matchups["Team1_WinPct"].isna()]
missing_rows

,Season,DayNum,WLoc,NumOT,Team1ID,Team2ID,Target,Team1_Season,Team1_TeamID,Team1_Wins,...,ReboundPctDiff,TurnoverPctDiff,FGPctDiff,ThreePctDiff,FTPctDiff,RankingDiff,PossessionsDiff,ReboundMarginDiff,TurnoverMarginDiff,AssistTurnoverRatioDiff


In [51]:
print("Training dataset shape:", men_training_df.shape)
print("Missing values:", men_training_df.isna().sum().sum())
print("Target distribution:")
print(men_training_df["Target"].value_counts(normalize=True))

Training dataset shape: (2898, 20)
Missing values: 0
Target distribution:
Target
1    0.5
0    0.5
Name: proportion, dtype: float64


In [52]:
men_training_df.to_csv("../data/m_tournament_training_dataset.csv", index=False)